<h1> Finding the API number associated with each leases <h1>

The data saved in pdq_lease_cycle.ipynb have the district no + lease number as the identifying information. But to get the coordinates of the wells, one need API number. OG_WELL_COMPLETION_DATA_TABLE.dsv links district no + lease no to API number. This program loads  this data into og_well_completion.parquet and saves it alongside the rest

In [1]:
# ── CELL 1: Imports + Config ──────────────────────────────────────────────────

import zipfile
import logging
import warnings
import sys
import gc
from io import TextIOWrapper, BytesIO
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger(__name__)

# ── Config — only edit these ──────────────────────────────────────────────────
OUTER_ZIP      = "../../../data/raw/texas/texas_pdq.zip"          # your outer zip
INNER_ZIP      = "PDQ_DSV.zip"                                    # zip inside outer
OUT_DIR        = "../../../data/raw/texas/pdq_lease_output"       # same folder as other parquets
FORMAT         = "parquet"
CHUNKSIZE      = 75_000
OIL_GAS_FILTER = "O"                                              # must match your production data

# Constants
DELIMITER  = "}"
ENCODING   = "latin-1"
WELL_FILE  = "OG_WELL_COMPLETION_DATA_TABLE.dsv"

# Only the columns we need — keeps memory very low
WELL_KEEP = [
    "OIL_GAS_CODE",
    "DISTRICT_NO",
    "LEASE_NO",       # joins to og_lease_cycle
    "WELL_NO",
    "API_COUNTY_CODE",
    "API_UNIQUE_NO",  # combined with API_COUNTY_CODE → API_NO → coordinates
    "COUNTY_NAME",
    "WELLBORE_LOCATION_CODE",
]

log.info("✓ Cell 1 done.")


14:57:28 [INFO] ✓ Cell 1 done.


In [2]:
# ── CELL 2: Cleaning + reader functions ───────────────────────────────────────

def _strip_cols(df):
    df.columns = df.columns.str.strip()
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())
    return df

def _apply_filter(df):
    if OIL_GAS_FILTER and "OIL_GAS_CODE" in df.columns:
        df = df[df["OIL_GAS_CODE"].str.strip() == OIL_GAS_FILTER]
    return df

def clean_well_chunk(chunk):
    chunk = _strip_cols(chunk)
    chunk = _apply_filter(chunk)
    if chunk.empty:
        return chunk
    chunk = chunk[[c for c in WELL_KEEP if c in chunk.columns]]
    # Build full 8-digit API number
    if "API_COUNTY_CODE" in chunk.columns and "API_UNIQUE_NO" in chunk.columns:
        chunk["API_NO"] = (
            chunk["API_COUNTY_CODE"].str.strip().str.zfill(3) +
            chunk["API_UNIQUE_NO"].str.strip().str.zfill(5)
        )
    for col in ("OIL_GAS_CODE", "DISTRICT_NO", "WELLBORE_LOCATION_CODE"):
        if col in chunk.columns:
            chunk[col] = chunk[col].astype("category")
    return chunk

def open_inner_zip(outer_path, inner_name):
    with zipfile.ZipFile(outer_path, "r") as outer:
        outer_contents = outer.namelist()
        log.info("Files in outer zip: %s", outer_contents)
        match = next((f for f in outer_contents
                      if f.upper() == inner_name.upper()), None)
        if match is None:
            raise FileNotFoundError(
                f"'{inner_name}' not found.\nAvailable: {outer_contents}")
        inner_bytes = BytesIO(outer.read(match))
    return zipfile.ZipFile(inner_bytes, "r")

def read_chunked(inner_zf, filename, cleaner):
    try:
        inner_zf.getinfo(filename)
    except KeyError:
        raise FileNotFoundError(
            f"'{filename}' not found.\nAvailable: {inner_zf.namelist()}")
    log.info("Reading %s (chunk size = %s) ...", filename, f"{CHUNKSIZE:,}")
    chunks, total_in, total_out = [], 0, 0
    with inner_zf.open(filename) as raw:
        reader = pd.read_csv(
            TextIOWrapper(raw, encoding=ENCODING),
            sep=DELIMITER,
            dtype=str,
            chunksize=CHUNKSIZE,
            low_memory=False,
            on_bad_lines="warn",
        )
        for i, chunk in enumerate(reader, 1):
            total_in += len(chunk)
            cleaned = cleaner(chunk)
            if not cleaned.empty:
                chunks.append(cleaned)
                total_out += len(cleaned)
            del chunk, cleaned
            gc.collect()
            log.info("  chunk %3d — kept %s / %s rows",
                     i, f"{total_out:,}", f"{total_in:,}")
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    log.info("✓ Done: %s rows x %s columns", f"{len(df):,}", len(df.columns))
    return df

log.info("✓ Cell 2 done — functions defined.")


11:33:42 [INFO] ✓ Cell 2 done — functions defined.


In [3]:
# ── CELL 3: Load OG_WELL_COMPLETION ──────────────────────────────────────────

log.info("\n── Loading OG_WELL_COMPLETION ──")
inner_zf = open_inner_zip(OUTER_ZIP, INNER_ZIP)
df_well  = read_chunked(inner_zf, WELL_FILE, clean_well_chunk)
inner_zf.close()
gc.collect()

print("\nShape  :", df_well.shape)
print("Memory :", f"{df_well.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print("\nColumns:", df_well.columns.tolist())
print("\nSample:")
print(df_well[["DISTRICT_NO", "LEASE_NO", "WELL_NO",
               "API_NO", "COUNTY_NAME"]].head(10))

# Quick check — how many have a valid API number
has_api = df_well["API_NO"].notna() & (df_well["API_NO"] != "00000000")
print(f"\nWells with valid API_NO: {has_api.sum():,} / {len(df_well):,}")


11:33:43 [INFO] 
── Loading OG_WELL_COMPLETION ──
11:33:43 [INFO] Files in outer zip: ['PDQ_DSV.zip']
11:33:48 [INFO] Reading OG_WELL_COMPLETION_DATA_TABLE.dsv (chunk size = 75,000) ...
11:33:49 [INFO]   chunk   1 — kept 0 / 75,000 rows
11:33:49 [INFO]   chunk   2 — kept 0 / 150,000 rows
11:33:49 [INFO]   chunk   3 — kept 495 / 225,000 rows
11:33:49 [INFO]   chunk   4 — kept 68,669 / 300,000 rows
11:33:49 [INFO]   chunk   5 — kept 143,669 / 375,000 rows
11:33:49 [INFO]   chunk   6 — kept 218,669 / 450,000 rows
11:33:49 [INFO]   chunk   7 — kept 293,669 / 525,000 rows
11:33:49 [INFO]   chunk   8 — kept 368,669 / 600,000 rows
11:33:49 [INFO]   chunk   9 — kept 443,669 / 675,000 rows
11:33:49 [INFO]   chunk  10 — kept 518,669 / 750,000 rows
11:33:50 [INFO]   chunk  11 — kept 587,614 / 818,945 rows
11:33:50 [INFO] ✓ Done: 587,614 rows x 9 columns

Shape  : (587614, 9)
Memory : 55.1 MB

Columns: ['OIL_GAS_CODE', 'DISTRICT_NO', 'LEASE_NO', 'WELL_NO', 'API_COUNTY_CODE', 'API_UNIQUE_NO', 'COUN

In [4]:
# ── CELL 4: Save ─────────────────────────────────────────────────────────────

out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

path = out / f"og_well_completion.{FORMAT}"
if FORMAT == "parquet":
    df_well.to_parquet(path, index=False)
else:
    df_well.to_csv(path, index=False)

size_mb = path.stat().st_size / 1_048_576
log.info("✓ Saved %s  (%.1f MB)", path.name, size_mb)

print("\nAll files in output folder:")
for f in sorted(out.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1_048_576:.1f} MB)")


11:33:53 [INFO] ✓ Saved og_well_completion.parquet  (6.4 MB)

All files in output folder:
  og_lease_cycle.parquet  (139.4 MB)
  og_lease_cycle_disp.parquet  (139.2 MB)
  og_well_completion.parquet  (6.4 MB)


In [5]:
# ── CELL 5: Quick join test ───────────────────────────────────────────────────
# Verifies that DISTRICT_NO + LEASE_NO in df_well
# matches what is in your production parquet.

log.info("Loading og_lease_cycle.parquet for join test ...")
df_cycle = pd.read_parquet(f"{OUT_DIR}/og_lease_cycle.parquet",
                            columns=["DISTRICT_NO", "LEASE_NO"])
gc.collect()

# Unique leases in production data
prod_leases = df_cycle.drop_duplicates(subset=["DISTRICT_NO", "LEASE_NO"])
log.info("Unique leases in production data : %s", f"{len(prod_leases):,}")

# Unique leases in well completion
well_leases = df_well.drop_duplicates(subset=["DISTRICT_NO", "LEASE_NO"])
log.info("Unique leases in well completion : %s", f"{len(well_leases):,}")

# How many production leases have at least one well with an API number
merged = prod_leases.merge(
    df_well[["DISTRICT_NO", "LEASE_NO", "API_NO"]].dropna(subset=["API_NO"]),
    on=["DISTRICT_NO", "LEASE_NO"],
    how="left"
)
matched = merged["API_NO"].notna().sum()
log.info("Production leases matched to an API number: %s / %s (%.1f%%)",
         f"{matched:,}", f"{len(prod_leases):,}",
         matched / len(prod_leases) * 100)

print("\n✓ Join test complete.")
print(f"  {matched:,} / {len(prod_leases):,} production leases have a matching API number.")
print("  These can be joined to coordinates via the permit master file.")

del df_cycle, prod_leases, well_leases, merged
gc.collect()


11:34:04 [INFO] Loading og_lease_cycle.parquet for join test ...
11:34:05 [INFO] Unique leases in production data : 163,070
11:34:05 [INFO] Unique leases in well completion : 169,866
11:34:05 [INFO] Production leases matched to an API number: 576,234 / 163,070 (353.4%)

✓ Join test complete.
  576,234 / 163,070 production leases have a matching API number.
  These can be joined to coordinates via the permit master file.


0